# 3- Trying the models

Classifier: SVM (linear), RF

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import cognitive_models.preprocessing as cwpre
import cognitive_models.features as cwfeat
import cognitive_models.interpolate as cwinterp
import cognitive_models.pupil_utils as cwpupil
from importlib import reload
reload(cwpupil)
reload(cwinterp)
reload(cwpre)
reload(cwfeat)

COLET_DATASET_DIR = Path(cwpre.__file__).parents[2] / "datasets" / "COLET_CSV"
PARTICIPANTS = list(range(1, 11)) # Participants 1 to 10
PARTICIPANTS.remove(6) # Bad data
EXPERIMENTS = [1,2,3,4]

### A. Getting all the needed features

In [2]:
# 1- Load all required data
all_eye_data = cwpre.load_colet_data(COLET_DATASET_DIR, PARTICIPANTS, EXPERIMENTS)

print(f"Loaded eye-tracking data for {len(PARTICIPANTS)} participants and {len(EXPERIMENTS)} experiments. Total records: {len(all_eye_data)}")

There are 1 non matching timestamp_sec within the limits
Final merged dataset has 8184 records at 240 Hz
Final merged and resampled dataset has 2028 records at 60 Hz
There are 1 non matching timestamp_sec within the limits
Final merged dataset has 6866 records at 240 Hz
Final merged and resampled dataset has 1709 records at 60 Hz
There are 0 non matching timestamp_sec within the limits
Final merged dataset has 17301 records at 240 Hz
Final merged and resampled dataset has 4285 records at 60 Hz
There are 0 non matching timestamp_sec within the limits
Final merged dataset has 9246 records at 240 Hz
Final merged and resampled dataset has 2290 records at 60 Hz
There are 0 non matching timestamp_sec within the limits
Final merged dataset has 10159 records at 240 Hz
Final merged and resampled dataset has 2505 records at 60 Hz
There are 0 non matching timestamp_sec within the limits
Final merged dataset has 7113 records at 240 Hz
Final merged and resampled dataset has 1769 records at 60 Hz
Th

In [3]:
# Lets check integrity of the data
all_eye_data.columns
all_eye_data.groupby(['subject_id', 'task_id'])['cl_class'].value_counts().sort_index().head()

subject_id  task_id  cl_class
1           1        low         2028
            2        medium      1709
            3        high        4285
            4        medium      2290
2           1        low         2505
Name: count, dtype: int64

In [4]:
import tqdm
all_eye_data_grouped = all_eye_data.groupby(['subject_id', 'task_id'])
# Extract all window features
Fs = 60
WINDOW_N = 512
WINDOW_STEP = 64
SKIP_EDGE_SAMPLES = Fs # first & last 2 seconds are discarded

features_df = pd.DataFrame()
for (subject_id, task_id), group_df in tqdm.tqdm(all_eye_data_grouped, desc="Extracting features"):
    feature_rows = []
    # Discard edge samples
    group_df = group_df.iloc[SKIP_EDGE_SAMPLES:-SKIP_EDGE_SAMPLES].sort_values('timestamp_sec')
    
    # Extract features in sliding windows
    for start in range(WINDOW_N, len(group_df), WINDOW_STEP):
        window_df = group_df.iloc[start - WINDOW_N:start].reset_index(drop=True)
        window_df, blink_df = cwpre.preprocess_colet_data(window_df, verbose=False) # Add derived columns
        features = cwfeat.extract_window_features(window_df, blink_df, ivt_threshold=45, min_fixation_duration=55, verbose=False)
        features['subject_id'] = subject_id
        features['task_id'] = task_id
        features['cl_class'] = window_df['cl_class'].iloc[0] # Class is the same for a given task
        feature_rows.append(features)
    
    features_df = pd.concat([features_df, pd.DataFrame(feature_rows)], ignore_index=True)

print(f"Extracted features for {len(features_df)} windows.")

Extracting features:   0%|          | 0/36 [00:00<?, ?it/s]

Extracting features: 100%|██████████| 36/36 [04:12<00:00,  7.03s/it]

Extracted features for 1213 windows.


In [5]:
# Check what is the issue with the NaNs
features_df[features_df.isna().any(axis=1)].head()

,fixations_count,fixations_duration_mean,fixations_duration_max,fixations_duration_min,fixations_duration_std,saccades_count,saccades_peak_velocity_mean,saccades_amplitude_mean,saccades_amplitude_max,saccades_amplitude_min,...,saccades_duration_mean,saccades_duration_max,saccades_duration_min,saccades_duration_std,blinks_count,blinks_duration_mean,pupil_lhipa,subject_id,task_id,cl_class


In [8]:
from sklearn import preprocessing
# z-normalize features
features_df_transformed = features_df.copy()
feature_cols = [col for col in features_df.columns if col not in ['subject_id', 'task_id', 'cl_class']]

features_df_transformed.describe()

,fixations_count,fixations_duration_mean,fixations_duration_max,fixations_duration_min,fixations_duration_std,saccades_count,saccades_peak_velocity_mean,saccades_amplitude_mean,saccades_amplitude_max,saccades_amplitude_min,...,saccades_duration_mean,saccades_duration_max,saccades_duration_min,saccades_duration_std,blinks_count,blinks_duration_mean,pupil_lhipa,subject_id,task_id,labels
count,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,...,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000,1213.000000
mean,14.488046,372.164205,1246.429505,80.230387,348.542211,15.991756,150.367888,1.798390,11.668994,0.037754,...,71.167968,290.742391,16.395144,78.592348,2.928277,0.111172,4.612794,5.421270,2.834295,1.028030
std,5.359600,147.318289,599.127409,47.228882,187.243416,4.576082,68.342090,2.393138,27.317539,0.079034,...,51.057762,308.249259,3.996759,92.953006,2.906230,0.068276,0.336892,2.922787,1.049003,0.837964
min,1.000000,100.020000,183.370000,50.010000,0.000000,3.000000,58.241446,0.268942,0.965858,0.000000,...,20.374444,33.340000,0.000000,7.350779,0.000000,0.000000,3.287014,1.000000,1.000000,0.000000
25%,10.000000,275.055000,833.500000,50.010000,219.105798,13.000000,99.779182,0.941384,2.210624,0.002450,...,41.675000,83.350000,16.670000,22.791500,1.000000,0.100020,4.343554,3.000000,2.000000,0.000000
50%,15.000000,340.264118,1100.220000,66.680000,300.357531,16.000000,133.082008,1.126361,2.547356,0.014155,...,53.714444,166.700000,16.670000,39.606347,2.000000,0.133360,4.578341,5.000000,3.000000,1.000000
75%,18.000000,425.575294,1550.310000,83.350000,422.330428,19.000000,178.710569,1.418602,4.743574,0.045167,...,82.369412,366.740000,16.670000,98.150302,4.000000,0.158365,4.813127,8.000000,4.000000,2.000000
max,29.000000,1455.846667,3867.440000,600.120000,1214.109414,28.000000,614.456017,24.943829,173.251822,1.369546,...,608.455000,1700.340000,50.010000,683.327680,16.000000,0.266720,5.752274,10.000000,4.000000,2.000000


In [10]:
from sklearn.model_selection import train_test_split
from sklearn import svm, pipeline

features_df['labels'] = features_df['cl_class'].map({'low': 0, 'medium': 1, 'high': 2})

X = features_df[feature_cols].values
y = features_df['labels'].values

scaler = preprocessing.StandardScaler()
scaler.fit(X)

train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42)
# Split train validation set from train set
train_x, val_x, train_y, val_y = train_test_split(train_x, train_y, test_size=0.25, random_state=42)

print(f"Training set size: {len(train_x)}, Validation set size: {len(val_x)}, Test set size: {len(test_x)}")

train_x_transformed = scaler.transform(train_x)
val_x_transformed = scaler.transform(val_x)
model = svm.LinearSVC()
model.fit(train_x_transformed, train_y)
train_acc = model.score(train_x_transformed, train_y)
val_acc = model.score(val_x_transformed, val_y)
print(f"Train Accuracy: {train_acc:.2f}, Validation Accuracy: {val_acc:.2f}")

Training set size: 727, Validation set size: 243, Test set size: 243
Train Accuracy: 1.00, Validation Accuracy: 1.00
